In [22]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline

In [23]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay, precision_score, recall_score, f1_score

In [24]:
np.random.seed(42)

n_normal = 950
n_anomalies = 50

normal_amount = np.random.normal(500, 120, n_normal)
normal_time = np.random.normal(12, 3, n_normal)

anomaly_amount = np.random.uniform(1200, 2500, n_anomalies)
anomaly_time = np.random.uniform(0, 24, n_anomalies)

amount = np.concatenate([normal_amount, anomaly_amount])
time = np.concatenate([normal_time, anomaly_time])

X = pd.DataFrame({
    "transaction_amount" : amount,
    "transaction_time" : time
})

y = np.concatenate([
    np.zeros(n_normal, dtype=int),
    np.ones(n_anomalies, dtype=int)
])

In [25]:
X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.25, random_state=42, stratify=y_train_val)

print("Training Rows : ", len(X_train))
print("Validation Rows : ", len(X_val))
print("Testing Rows : ", len(X_test))


Training Rows :  600
Validation Rows :  200
Testing Rows :  200


In [28]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

contamination_values = [0.01, 0.03, 0.05, 0.08, 0.10]

model_configs = {
    "Isolation Forest" : {
        "kind" : "iforest",
        "values" : contamination_values
    }, 
    "LOF" : {
        "kind" : "lof",
        "values" : contamination_values
    },
    "One-Class SVM" : {
        "kind" : "svm",
        "values" : contamination_values
    }
}

def create_model(kind, value) : 
    if kind == 'iforest':
        return IsolationForest(
            n_estimators=100,
            contamination=value,
            random_state=42
        )
    if kind == 'lof':
        return LocalOutlierFactor(
            n_neighbors=20,
            contamination=value,
            novelty=True
        )
    if kind == 'svm':
        return OneClassSVM(
            kernel='rbf',
            gamma="scale",
            nu=value
        )

def predict_anomalies(model, X_data):
    return (model.predict(X_data) == -1).astype(int)

In [29]:
best_settings = {}
validation_results = []

for model_name, config in model_configs.items():
    best_f1 = -1
    best_value = None
    best_precision = 0
    best_recall = 0

    for value in config['values']:
        model = create_model(config['kind'], value)
        model.fit(X_train_scaled)

        val_pred = predict_anomalies(model, X_val_scaled)

        precision = precision_score(y_val, val_pred, zero_division=0)
        recall = recall_score(y_val, val_pred, zero_division=0)
        f1 = f1_score(y_val, val_pred, zero_division=0)

        validation_results.append({
            "model" : model_name,
            "setting" : value,
            "precision" : precision,
            "recall" : recall,
            "f1 score" : f1
        })

        if f1 > best_f1:
            best_f1 = f1
            best_value = value
            best_precision = precision
            best_recall = recall

    best_settings[model_name] = {
        "kind" : config["kind"],
        "value" : best_value
    }

print("VALIDATION RESULTS")
print(pd.DataFrame(validation_results).round(3).to_string(index=False))
for name, setting in best_settings.items():
    print(name, ":", setting['value'])

VALIDATION RESULTS
           model  setting  precision  recall  f1 score
Isolation Forest     0.01      1.000     0.5     0.667
Isolation Forest     0.03      1.000     0.9     0.947
Isolation Forest     0.05      1.000     1.0     1.000
Isolation Forest     0.08      0.556     1.0     0.714
Isolation Forest     0.10      0.476     1.0     0.645
             LOF     0.01      0.500     0.1     0.167
             LOF     0.03      0.600     0.3     0.400
             LOF     0.05      0.364     0.4     0.381
             LOF     0.08      0.267     0.4     0.320
             LOF     0.10      0.182     0.4     0.250
   One-Class SVM     0.01      0.400     0.4     0.400
   One-Class SVM     0.03      0.400     0.4     0.400
   One-Class SVM     0.05      0.500     0.7     0.583
   One-Class SVM     0.08      0.562     0.9     0.692
   One-Class SVM     0.10      0.526     1.0     0.690
Isolation Forest : 0.05
LOF : 0.03
One-Class SVM : 0.08
